# Phased Array Antennas — Wavefront Propagation & Beam Control

A **phased array** steers its beam electronically by applying progressive time delays (or phase shifts) across identical antenna elements — no mechanical rotation needed.

## Key ideas

| Concept | Description |
|---------|-------------|
| Array Factor (AF) | Spatial interference pattern of $N$ isotropic sources |
| Progressive phase $\delta$ | Phase increment between adjacent elements to steer the beam |
| Beam steering | $\delta = -kd\sin\theta_0$ points the main lobe at $\theta_0$ |
| Grating lobes | Parasitic main lobes when $d > \lambda/(1+|\sin\theta_0|)$ |
| True time delay (TTD) | Broadband steering: delay $\tau_n = n d \sin\theta_0 / c$ |

## Array Factor for a Uniform Linear Array

$$AF(\theta) = \sum_{n=0}^{N-1} w_n\, e^{\,j n (k d \sin\theta + \delta)}$$

For uniform weights $w_n = 1/N$:
$$|AF(\theta)| = \frac{1}{N}\left|\frac{\sin\!\bigl(\tfrac{N\psi}{2}\bigr)}{\sin\!\bigl(\tfrac{\psi}{2}\bigr)}\right|, \qquad \psi = kd\sin\theta + \delta$$

This notebook focuses on **physical wavefront propagation**, **E / H field visualisation**, and the distinction between **phase-shift** and **true-time-delay** beam steering.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from ipywidgets import (
    FloatSlider, IntSlider, Dropdown, Checkbox,
    HBox, VBox, Output, Label, Play, jslink
)
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- shared constants & helpers ----------
c0 = 3e8  # speed of light

def af_pattern(theta, N, d_lam, steer_rad):
    """Normalised AF for ULA, broadside = 0°."""
    k = 2 * np.pi
    delta = -k * d_lam * np.sin(steer_rad)
    psi = k * d_lam * np.sin(theta) + delta
    with np.errstate(divide='ignore', invalid='ignore'):
        af = np.where(np.abs(np.sin(psi / 2)) < 1e-12, 1.0,
                      np.abs(np.sin(N * psi / 2) / (N * np.sin(psi / 2))))
    return af

def sinc_safe(x):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(np.abs(x) < 1e-10, 1.0, np.sin(x) / x)

## 1 — Element Positions & Wavefront Propagation

Each element radiates a spherical wavefront.  
A progressive **time delay** $\tau_n = n\,\Delta\tau$ makes the wavefronts add up coherently at the steered angle $\theta_0$:
$$\Delta\tau = \frac{d\sin\theta_0}{c}$$

The slider below controls:
- **N** — number of elements
- **d/λ** — element spacing in wavelengths  
- **θ₀** — beam steering angle  
- **t** — time snapshot (watch wavefronts propagate and combine)

In [ ]:
w_wN   = IntSlider(value=6, min=2, max=16, step=1,
                   description='Elements N', continuous_update=False,
                   style={'description_width': '90px'})
w_wd   = FloatSlider(value=0.5, min=0.2, max=1.5, step=0.05,
                     description='Spacing d/λ', continuous_update=False,
                     style={'description_width': '90px'})
w_wth  = FloatSlider(value=0.0, min=-80, max=80, step=1,
                     description='Steer θ₀ (°)', continuous_update=False,
                     style={'description_width': '90px'})
w_wt   = FloatSlider(value=0.0, min=0.0, max=3.0, step=0.05,
                     description='Time t/T', continuous_update=False,
                     style={'description_width': '90px'})
out_wf = Output()

def draw_wavefronts(change=None):
    N = w_wN.value;  d = w_wd.value
    th0 = np.radians(w_wth.value)
    t_norm = w_wt.value          # time in periods
    lam = 1.0;  k = 2 * np.pi / lam
    omega = 2 * np.pi            # period = 1

    # element positions along y-axis
    ys = (np.arange(N) - (N - 1) / 2) * d
    delays = -np.arange(N) * d * np.sin(th0) / lam  # in periods

    # observation grid
    xg = np.linspace(-0.5, 8, 400)
    yg = np.linspace(ys.min() - 3, ys.max() + 3, 400)
    X, Y = np.meshgrid(xg, yg)

    field = np.zeros_like(X)
    for yi, tau in zip(ys, delays):
        R = np.sqrt(X**2 + (Y - yi)**2) + 1e-12
        field += np.cos(omega * (t_norm - tau) - k * R) / np.sqrt(R)

    with out_wf:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6),
                                        gridspec_kw={'width_ratios': [2, 1]})

        # --- wavefront map ---
        vmax = np.percentile(np.abs(field), 97)
        ax1.pcolormesh(X, Y, field, cmap='RdBu_r', shading='auto',
                       vmin=-vmax, vmax=vmax, rasterized=True)
        # element markers
        ax1.scatter(np.zeros(N), ys, marker='|', s=300, c='lime',
                    linewidths=2, zorder=5, label='Elements')
        # steering direction arrow
        arr_len = 3
        ax1.annotate('', xy=(arr_len * np.cos(th0), arr_len * np.sin(th0)),
                     xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color='yellow', lw=2))
        ax1.set_xlabel('x / λ');  ax1.set_ylabel('y / λ')
        ax1.set_title(f'Wavefront propagation   N={N}  d={d:.2f}λ  '
                      f'θ₀={w_wth.value:.0f}°  t={t_norm:.2f}T')
        ax1.set_aspect('equal')
        ax1.legend(loc='upper right', fontsize=8)

        # --- element delay diagram ---
        colours = plt.cm.viridis(np.linspace(0.2, 0.9, N))
        ax2.barh(np.arange(N), delays, color=colours, height=0.6)
        ax2.set_yticks(np.arange(N))
        ax2.set_yticklabels([f'Elem {i}' for i in range(N)], fontsize=8)
        ax2.set_xlabel('Delay  τₙ / T')
        ax2.set_title('Per-element delays')
        ax2.grid(alpha=0.3)
        ax2.invert_yaxis()

        plt.tight_layout();  plt.show()

for w in [w_wN, w_wd, w_wth, w_wt]:
    w.observe(draw_wavefronts, names='value')
draw_wavefronts()
display(VBox([HBox([w_wN, w_wd]), HBox([w_wth, w_wt]), out_wf]))

## 2 — E-field & H-field Spatial Maps

An EM plane wave has orthogonal **E** and **H** components.  
For a vertically polarised array (elements along $y$), the dominant field components in the far zone are:

$$E_z(\mathbf{r}, t) = \sum_n \frac{w_n}{\sqrt{r_n}}\cos\!\bigl(\omega(t - \tau_n) - k r_n\bigr)$$

$$H_\perp = \frac{E_z}{\eta_0}, \qquad \eta_0 = 377\,\Omega$$

The **H-field** is perpendicular to both propagation direction and **E**.  
Below: spatial snapshots of $E_z$ and $H_y$ from the steered array.

In [3]:
w_eN   = IntSlider(value=8,   min=2, max=16, step=1,
                   description='Elements N', continuous_update=False,
                   style={'description_width': '90px'})
w_ed   = FloatSlider(value=0.5, min=0.2, max=1.5, step=0.05,
                     description='Spacing d/λ', continuous_update=False,
                     style={'description_width': '90px'})
w_eth  = FloatSlider(value=20, min=-80, max=80, step=1,
                     description='Steer θ₀ (°)', continuous_update=False,
                     style={'description_width': '90px'})
w_et   = FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05,
                     description='Time t/T', continuous_update=False,
                     style={'description_width': '90px'})
out_eh = Output()

def draw_EH(change=None):
    N = w_eN.value;  d = w_ed.value
    th0 = np.radians(w_eth.value);  t_norm = w_et.value
    lam = 1.0;  k = 2 * np.pi / lam;  omega = 2 * np.pi

    ys = (np.arange(N) - (N - 1) / 2) * d
    delays = -np.arange(N) * d * np.sin(th0) / lam

    xg = np.linspace(0.2, 10, 350)
    yg = np.linspace(ys.min() - 3, ys.max() + 3, 350)
    X, Y = np.meshgrid(xg, yg)

    Ez = np.zeros_like(X)
    Hy = np.zeros_like(X)
    for yi, tau in zip(ys, delays):
        R = np.sqrt(X**2 + (Y - yi)**2) + 1e-12
        cos_phi = X / R          # direction cosine for H projection
        phase = omega * (t_norm - tau) - k * R
        amp = 1.0 / np.sqrt(R)
        Ez += amp * np.cos(phase)
        Hy += amp * np.cos(phase) * cos_phi  # H ~ E × r̂, project onto y

    with out_eh:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

        vE = np.percentile(np.abs(Ez), 97)
        vH = np.percentile(np.abs(Hy), 97)

        im1 = ax1.pcolormesh(X, Y, Ez, cmap='RdBu_r', shading='auto',
                             vmin=-vE, vmax=vE, rasterized=True)
        ax1.scatter(np.zeros(N), ys, marker='|', s=200, c='lime', linewidths=2, zorder=5)
        ax1.set_title(f'E_z field   θ₀={w_eth.value:.0f}°  t={t_norm:.2f}T')
        ax1.set_xlabel('x / λ');  ax1.set_ylabel('y / λ')
        ax1.set_aspect('equal')
        plt.colorbar(im1, ax=ax1, fraction=0.046, label='E_z')

        im2 = ax2.pcolormesh(X, Y, Hy, cmap='PiYG', shading='auto',
                             vmin=-vH, vmax=vH, rasterized=True)
        ax2.scatter(np.zeros(N), ys, marker='|', s=200, c='lime', linewidths=2, zorder=5)
        ax2.set_title(f'H_y field   θ₀={w_eth.value:.0f}°  t={t_norm:.2f}T')
        ax2.set_xlabel('x / λ')
        ax2.set_aspect('equal')
        plt.colorbar(im2, ax=ax2, fraction=0.046, label='H_y')

        plt.tight_layout();  plt.show()

for w in [w_eN, w_ed, w_eth, w_et]:
    w.observe(draw_EH, names='value')
draw_EH()
display(VBox([HBox([w_eN, w_ed]), HBox([w_eth, w_et]), out_eh]))

## 3 — Sliding Elements: Spacing Effect on Beam

Drag the **d/λ** slider to "slide" elements apart or together and watch how the radiation pattern and grating lobes evolve in real time.

- $d < \lambda/2$: wide main lobe, no grating lobes  
- $d = \lambda/2$: classic design point  
- $d > \lambda/2$: grating lobes enter visible space  

The **left** plot shows element positions and wavefronts at a fixed time;  
the **right** plot shows the polar radiation pattern (AF × element factor).

In [4]:
w_sN  = IntSlider(value=8,   min=2,  max=20,  step=1,
                  description='Elements N', continuous_update=False,
                  style={'description_width': '90px'})
w_sd  = FloatSlider(value=0.5, min=0.15, max=2.0, step=0.05,
                    description='Spacing d/λ', continuous_update=False,
                    style={'description_width': '90px'})
w_sth = FloatSlider(value=0.0, min=-80, max=80, step=1,
                    description='Steer θ₀ (°)', continuous_update=False,
                    style={'description_width': '90px'})
out_slide = Output()

theta_full = np.linspace(0, 2 * np.pi, 3600)

def draw_sliding(change=None):
    N = w_sN.value;  d = w_sd.value
    th0 = np.radians(w_sth.value)
    lam = 1.0;  k = 2 * np.pi / lam

    ys = (np.arange(N) - (N - 1) / 2) * d
    delays = -np.arange(N) * d * np.sin(th0) / lam

    # AF pattern
    delta = -k * d * np.sin(th0)
    psi = k * d * np.sin(theta_full) + delta
    with np.errstate(divide='ignore', invalid='ignore'):
        AF = np.where(np.abs(np.sin(psi / 2)) < 1e-12, 1.0,
                      np.abs(np.sin(N * psi / 2) / (N * np.sin(psi / 2))))

    # simple element factor (short dipole)
    EF = np.abs(np.cos(theta_full))
    EF /= EF.max() + 1e-30
    TOT = AF * EF;  TOT /= TOT.max() + 1e-30

    # wavefront snapshot at t=0
    xg = np.linspace(-0.5, 5, 250)
    yg_grid = np.linspace(ys.min() - 2, ys.max() + 2, 250)
    X, Y = np.meshgrid(xg, yg_grid)
    field = np.zeros_like(X)
    for yi, tau in zip(ys, delays):
        R = np.sqrt(X**2 + (Y - yi)**2) + 1e-12
        field += np.cos(2 * np.pi * (-tau) - k * R) / np.sqrt(R)

    # grating lobe angles
    gl_angles = []
    for m in range(-5, 6):
        if m == 0:
            continue
        sin_val = np.sin(th0) + m * lam / d
        if -1 <= sin_val <= 1:
            gl_angles.append(np.degrees(np.arcsin(sin_val)))

    with out_slide:
        clear_output(wait=True)
        fig = plt.figure(figsize=(14, 6))
        ax1 = fig.add_subplot(121)
        ax2 = fig.add_subplot(122, projection='polar')

        # wavefront map with element positions
        vmax = np.percentile(np.abs(field), 95)
        ax1.pcolormesh(X, Y, field, cmap='RdBu_r', shading='auto',
                       vmin=-vmax, vmax=vmax, rasterized=True)
        ax1.scatter(np.zeros(N), ys, marker='s', s=80, c='lime',
                    edgecolors='k', linewidths=1, zorder=5)
        for i, yi in enumerate(ys):
            ax1.annotate(f'{i}', (0.15, yi), fontsize=7, color='white',
                         fontweight='bold')
        ax1.set_xlabel('x / λ');  ax1.set_ylabel('y / λ')
        ax1.set_title(f'Element layout & wavefronts   d={d:.2f}λ')
        ax1.set_aspect('equal')

        # polar pattern
        ax2.plot(theta_full, TOT, lw=2, color='gold', label='Total')
        ax2.plot(theta_full, AF,  lw=1, color='tomato', alpha=0.5, label='AF')
        ax2.set_ylim(0, 1.05)
        gl_text = f'  GL: {", ".join(f"{a:.0f}°" for a in gl_angles)}' if gl_angles else '  No grating lobes'
        ax2.set_title(f'Pattern  θ₀={w_sth.value:.0f}°{gl_text}', pad=12)
        ax2.legend(loc='upper right', fontsize=8)

        plt.tight_layout();  plt.show()

for w in [w_sN, w_sd, w_sth]:
    w.observe(draw_sliding, names='value')
draw_sliding()
display(VBox([HBox([w_sN, w_sd, w_sth]), out_slide]))

## 4 — Phase-Shift vs True-Time-Delay Steering

Two methods to steer a phased array:

| Method | Delay at element $n$ | Bandwidth |
|--------|---------------------|-----------|
| **Phase shifter** | $\phi_n = -n k_0 d \sin\theta_0$ (fixed at $f_0$) | Narrow — beam squints with frequency |
| **True time delay** | $\tau_n = n d \sin\theta_0 / c$ | Wide — steering angle is frequency-independent |

**Beam squint**: with phase shifters, the steered angle drifts as frequency moves from the design frequency $f_0$:
$$\theta(f) = \arcsin\!\left(\frac{f_0}{f}\sin\theta_0\right)$$

Below: compare patterns at different frequencies for both methods.

In [5]:
w_pN   = IntSlider(value=16,  min=4,  max=32,  step=1,
                   description='Elements N', continuous_update=False,
                   style={'description_width': '90px'})
w_pd   = FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05,
                     description='d/λ₀', continuous_update=False,
                     style={'description_width': '90px'})
w_pth  = FloatSlider(value=30.0, min=5, max=70, step=1,
                     description='Steer θ₀ (°)', continuous_update=False,
                     style={'description_width': '90px'})
w_pf   = FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05,
                     description='f / f₀', continuous_update=False,
                     style={'description_width': '90px'})
out_ps = Output()

th_scan = np.linspace(-90, 90, 3600)
th_scan_rad = np.radians(th_scan)

def draw_phase_vs_ttd(change=None):
    N = w_pN.value;  d0 = w_pd.value
    th0 = np.radians(w_pth.value)
    f_ratio = w_pf.value  # f / f0

    k0 = 2 * np.pi  # at f0, lambda0 = 1
    k  = k0 * f_ratio  # at f
    ns = np.arange(N)

    # --- Phase-shift steering (designed at f0) ---
    phase_shifts = -ns * k0 * d0 * np.sin(th0)  # fixed phases
    AF_ps = np.abs(np.sum(
        np.exp(1j * (ns[:, None] * k * d0 * np.sin(th_scan_rad)[None, :] + phase_shifts[:, None])),
        axis=0)) / N

    # --- True-time-delay steering ---
    ttd_delays = ns * d0 * np.sin(th0)  # in wavelengths at f0, = tau * c / lambda0
    AF_ttd = np.abs(np.sum(
        np.exp(1j * k * (ns[:, None] * d0 * np.sin(th_scan_rad)[None, :] - ttd_delays[:, None])),
        axis=0)) / N

    # beam squint prediction
    sin_squint = np.sin(th0) / f_ratio
    squint_ang = np.degrees(np.arcsin(np.clip(sin_squint, -1, 1)))

    AF_ps_dB  = 20 * np.log10(AF_ps  + 1e-10)
    AF_ttd_dB = 20 * np.log10(AF_ttd + 1e-10)

    with out_ps:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

        ax1.plot(th_scan, AF_ps_dB, lw=2, color='tomato', label='Phase shifter')
        ax1.axvline(w_pth.value, color='gray', ls=':', lw=1, label=f'Design θ₀={w_pth.value:.0f}°')
        ax1.axvline(squint_ang, color='gold', ls='--', lw=1.5, label=f'Actual peak ≈ {squint_ang:.1f}°')
        ax1.set_xlim(-90, 90);  ax1.set_ylim(-40, 3)
        ax1.set_xlabel('θ (°)');  ax1.set_ylabel('AF (dB)')
        ax1.set_title(f'Phase-shift steering   f/f₀={f_ratio:.2f}')
        ax1.legend(fontsize=8);  ax1.grid(alpha=0.3)

        ax2.plot(th_scan, AF_ttd_dB, lw=2, color='steelblue', label='True time delay')
        ax2.axvline(w_pth.value, color='gray', ls=':', lw=1, label=f'θ₀={w_pth.value:.0f}°')
        ax2.set_xlim(-90, 90)
        ax2.set_xlabel('θ (°)')
        ax2.set_title(f'True-time-delay steering   f/f₀={f_ratio:.2f}')
        ax2.legend(fontsize=8);  ax2.grid(alpha=0.3)

        fig.suptitle(f'Beam squint: phase shift → {squint_ang:.1f}°   TTD → {w_pth.value:.0f}° (no squint)',
                     fontsize=11, y=1.02)
        plt.tight_layout();  plt.show()

for w in [w_pN, w_pd, w_pth, w_pf]:
    w.observe(draw_phase_vs_ttd, names='value')
draw_phase_vs_ttd()
display(VBox([HBox([w_pN, w_pd]), HBox([w_pth, w_pf]), out_ps]))

## 5 — Near-Field Beam Formation

In the **near field** ($R < 2D^2/\lambda$, where $D$ = array aperture) the beam has a finite width and the wavefronts are still curving.  
The **far field** is the region where the pattern shape stabilises.

Below: intensity map $|E|^2$ showing how the beam forms as it propagates.  
Dashed line = Fraunhofer boundary $R_{ff} = 2D^2/\lambda$.

In [6]:
w_nN  = IntSlider(value=12,  min=4,  max=24, step=1,
                  description='Elements N', continuous_update=False,
                  style={'description_width': '90px'})
w_nd  = FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05,
                    description='Spacing d/λ', continuous_update=False,
                    style={'description_width': '90px'})
w_nth = FloatSlider(value=0.0, min=-60, max=60, step=1,
                    description='Steer θ₀ (°)', continuous_update=False,
                    style={'description_width': '90px'})
out_nf = Output()

def draw_nearfield(change=None):
    N = w_nN.value;  d = w_nd.value
    th0 = np.radians(w_nth.value)
    lam = 1.0;  k = 2 * np.pi / lam

    ys = (np.arange(N) - (N - 1) / 2) * d
    delays = -np.arange(N) * d * np.sin(th0) / lam
    D_ap = (N - 1) * d  # aperture
    R_ff = 2 * D_ap**2 / lam

    x_max = max(R_ff * 1.5, 10)
    xg = np.linspace(0.1, x_max, 400)
    yg = np.linspace(ys.min() - D_ap, ys.max() + D_ap, 400)
    X, Y = np.meshgrid(xg, yg)

    field = np.zeros_like(X, dtype=complex)
    for yi, tau in zip(ys, delays):
        R = np.sqrt(X**2 + (Y - yi)**2) + 1e-12
        field += np.exp(1j * (2 * np.pi * (-tau) - k * R)) / np.sqrt(R)

    intensity = np.abs(field)**2
    intensity /= intensity.max() + 1e-30

    with out_nf:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 6))

        im = ax.pcolormesh(X, Y, 10 * np.log10(intensity + 1e-10),
                           cmap='hot', shading='auto', vmin=-30, vmax=0,
                           rasterized=True)
        ax.axvline(R_ff, color='cyan', ls='--', lw=1.5,
                   label=f'Fraunhofer R_ff = {R_ff:.1f}λ')
        ax.scatter(np.zeros(N), ys, marker='|', s=200, c='lime',
                   linewidths=2, zorder=5)

        # steering direction
        arr_len = x_max * 0.4
        ax.annotate('', xy=(arr_len * np.cos(th0), arr_len * np.sin(th0)),
                     xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color='yellow', lw=2))

        ax.set_xlabel('x / λ');  ax.set_ylabel('y / λ')
        ax.set_title(f'Near-field intensity  N={N}  d={d:.2f}λ  '
                      f'θ₀={w_nth.value:.0f}°   D={D_ap:.1f}λ   R_ff={R_ff:.1f}λ')
        ax.set_aspect('equal')
        ax.legend(loc='upper right')
        plt.colorbar(im, ax=ax, fraction=0.046, label='|E|² (dB)')
        plt.tight_layout();  plt.show()

for w in [w_nN, w_nd, w_nth]:
    w.observe(draw_nearfield, names='value')
draw_nearfield()
display(VBox([HBox([w_nN, w_nd, w_nth]), out_nf]))

## 6 — Grating Lobes: When & Where They Appear

Grating lobes are full-amplitude replicas of the main beam.  
For beam steered to $\theta_0$ with spacing $d$, grating lobes appear at:
$$\sin\theta_{GL} = \sin\theta_0 + \frac{m\lambda}{d}, \quad m = \pm1, \pm2, \ldots$$
whenever $|\sin\theta_{GL}| \le 1$ (i.e. inside visible space).

**Design rule**: $d \le \frac{\lambda}{1 + |\sin\theta_{\max}|}$ ensures no grating lobes over the full scan range.

Below: the $d/\lambda$ – $\theta_0$ plane.  Coloured regions show where grating lobes exist.  
The interactive plot updates the pattern as you change spacing and steering.

In [7]:
w_gN  = IntSlider(value=10,  min=4,  max=24,  step=1,
                  description='Elements N', continuous_update=False,
                  style={'description_width': '90px'})
w_gd  = FloatSlider(value=0.7, min=0.2, max=2.0, step=0.05,
                    description='d/λ', continuous_update=False,
                    style={'description_width': '90px'})
w_gth = FloatSlider(value=20.0, min=-80, max=80, step=1,
                    description='Steer θ₀ (°)', continuous_update=False,
                    style={'description_width': '90px'})
out_gl = Output()

def draw_grating(change=None):
    N = w_gN.value;  d = w_gd.value
    th0 = np.radians(w_gth.value)

    # AF pattern
    th_plot = np.linspace(-90, 90, 3600)
    th_plot_rad = np.radians(th_plot)
    AF = af_pattern(th_plot_rad, N, d, th0)
    AF_dB = 20 * np.log10(AF + 1e-10)

    # grating lobe map: d vs theta0
    d_range = np.linspace(0.2, 2.0, 300)
    th0_range = np.linspace(-80, 80, 300)
    D_grid, TH0_grid = np.meshgrid(d_range, th0_range)
    n_gl = np.zeros_like(D_grid)
    for m in range(-3, 4):
        if m == 0:
            continue
        sin_gl = np.sin(np.radians(TH0_grid)) + m / D_grid
        n_gl += (np.abs(sin_gl) <= 1).astype(float)

    # grating lobe angles for current settings
    gl_angles = []
    for m in range(-5, 6):
        if m == 0:
            continue
        sin_val = np.sin(th0) + m / d
        if -1 <= sin_val <= 1:
            gl_angles.append(np.degrees(np.arcsin(sin_val)))

    with out_gl:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

        # grating lobe map
        im = ax1.pcolormesh(d_range, th0_range, n_gl, cmap='YlOrRd',
                            shading='auto', vmin=0, vmax=6)
        # no-grating-lobe boundary
        th_boundary = np.linspace(-80, 80, 300)
        d_boundary = 1.0 / (1 + np.abs(np.sin(np.radians(th_boundary))))
        ax1.plot(d_boundary, th_boundary, 'c--', lw=2, label='No-GL boundary')
        ax1.scatter([d], [w_gth.value], s=120, c='lime', edgecolors='k',
                    zorder=5, label=f'd={d:.2f}λ, θ₀={w_gth.value:.0f}°')
        ax1.set_xlabel('d / λ');  ax1.set_ylabel('θ₀ (°)')
        ax1.set_title('Grating lobe count in visible space')
        ax1.legend(fontsize=8)
        plt.colorbar(im, ax=ax1, label='# grating lobes')

        # pattern
        ax2.plot(th_plot, AF_dB, lw=2, color='gold')
        ax2.axvline(w_gth.value, color='gray', ls=':', lw=1)
        for gl in gl_angles:
            ax2.axvline(gl, color='tomato', ls='--', lw=1, alpha=0.7)
        ax2.set_xlim(-90, 90);  ax2.set_ylim(-40, 3)
        ax2.set_xlabel('θ (°)');  ax2.set_ylabel('AF (dB)')
        gl_str = ', '.join(f'{a:.0f}°' for a in gl_angles) if gl_angles else 'none'
        ax2.set_title(f'AF pattern   grating lobes at: {gl_str}')
        ax2.grid(alpha=0.3)

        plt.tight_layout();  plt.show()

for w in [w_gN, w_gd, w_gth]:
    w.observe(draw_grating, names='value')
draw_grating()
display(VBox([HBox([w_gN, w_gd, w_gth]), out_gl]))

## 7 — Beam Scanning & Array Performance Metrics

As the beam scans away from broadside, several things happen:
- **HPBW** broadens: $\text{HPBW}(\theta_0) \approx \text{HPBW}_0 / \cos\theta_0$
- **Directivity** decreases
- **Scan loss** ≈ $\cos\theta_0$ for element pattern roll-off

The left plot sweeps the beam from $-90°$ to $+90°$ and stacks patterns.  
The right plot tracks HPBW and peak gain vs. scan angle.

In [8]:
w_bN = IntSlider(value=16, min=4, max=32, step=1,
                 description='Elements N', continuous_update=False,
                 style={'description_width': '90px'})
w_bd = FloatSlider(value=0.5, min=0.25, max=0.75, step=0.05,
                   description='d/λ', continuous_update=False,
                   style={'description_width': '90px'})
out_bscan = Output()

def draw_beamscan(change=None):
    N = w_bN.value;  d = w_bd.value
    th_plot = np.linspace(-90, 90, 1800)
    th_plot_rad = np.radians(th_plot)

    scan_angles = np.arange(-80, 81, 5)
    hpbws = []
    peaks = []

    with out_bscan:
        clear_output(wait=True)
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

        # waterfall of patterns
        patterns = np.zeros((len(scan_angles), len(th_plot)))
        for i, th0_deg in enumerate(scan_angles):
            th0 = np.radians(th0_deg)
            AF = af_pattern(th_plot_rad, N, d, th0)
            EF = np.abs(np.cos(th_plot_rad))
            EF /= EF.max() + 1e-30
            pat = AF * EF
            pat /= pat.max() + 1e-30
            patterns[i] = pat

            # metrics
            pat_dB = 20 * np.log10(pat + 1e-10)
            hp = np.where(pat_dB >= -3)[0]
            hpbw = th_plot[hp[-1]] - th_plot[hp[0]] if len(hp) >= 2 else np.nan
            hpbws.append(hpbw)
            peaks.append(pat.max())

        pat_dB_map = 20 * np.log10(patterns + 1e-10)
        im = ax1.pcolormesh(th_plot, scan_angles, pat_dB_map,
                            cmap='inferno', shading='auto', vmin=-30, vmax=0)
        ax1.set_xlabel('θ (°)');  ax1.set_ylabel('Scan angle θ₀ (°)')
        ax1.set_title('Pattern vs. scan angle')
        plt.colorbar(im, ax=ax1, label='dB')

        ax2.plot(scan_angles, hpbws, 'o-', color='steelblue', markersize=3)
        # theoretical HPBW broadening
        cos_arr = np.abs(np.cos(np.radians(scan_angles)))
        cos_arr = np.where(cos_arr < 0.05, np.nan, cos_arr)
        hpbw0 = hpbws[len(scan_angles) // 2] if not np.isnan(hpbws[len(scan_angles) // 2]) else 10
        ax2.plot(scan_angles, hpbw0 / cos_arr, '--', color='tomato',
                 label=f'HPBW₀/cos(θ₀)', alpha=0.7)
        ax2.set_xlabel('Scan angle θ₀ (°)');  ax2.set_ylabel('HPBW (°)')
        ax2.set_title('Beamwidth vs. scan angle')
        ax2.legend(fontsize=8);  ax2.grid(alpha=0.3)

        scan_loss_dB = 10 * np.log10(np.abs(np.cos(np.radians(scan_angles))) + 1e-10)
        ax3.plot(scan_angles, scan_loss_dB, 'o-', color='gold', markersize=3)
        ax3.set_xlabel('Scan angle θ₀ (°)');  ax3.set_ylabel('Scan loss (dB)')
        ax3.set_title('cos(θ₀) scan loss')
        ax3.grid(alpha=0.3)

        plt.tight_layout();  plt.show()

for w in [w_bN, w_bd]:
    w.observe(draw_beamscan, names='value')
draw_beamscan()
display(VBox([HBox([w_bN, w_bd]), out_bscan]))

## 8 — Planar Array: 2-D Steering with E & H Cuts

An $N_x \times N_y$ planar array can steer in both azimuth and elevation.  
Using direction cosines $u = \sin\theta\cos\phi$, $v = \sin\theta\sin\phi$:

$$AF_{2D}(u,v) = \left|\sum_{m=0}^{N_x-1}\sum_{n=0}^{N_y-1} e^{\,jk(m d_x (u-u_0) + n d_y (v-v_0))}\right|$$

Below: heat map in $(u,v)$ space with E-plane and H-plane cuts overlaid.

In [9]:
w_2Nx = IntSlider(value=8,   min=2,  max=20,  step=1,
                  description='Nx', continuous_update=False,
                  style={'description_width': '40px'})
w_2Ny = IntSlider(value=8,   min=2,  max=20,  step=1,
                  description='Ny', continuous_update=False,
                  style={'description_width': '40px'})
w_2dx = FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05,
                    description='dx/λ', continuous_update=False,
                    style={'description_width': '40px'})
w_2dy = FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05,
                    description='dy/λ', continuous_update=False,
                    style={'description_width': '40px'})
w_2u0 = FloatSlider(value=0.0, min=-0.8, max=0.8, step=0.05,
                    description='u₀', continuous_update=False,
                    style={'description_width': '40px'})
w_2v0 = FloatSlider(value=0.0, min=-0.8, max=0.8, step=0.05,
                    description='v₀', continuous_update=False,
                    style={'description_width': '40px'})
out_2d = Output()

N_UV = 250
u1d  = np.linspace(-1, 1, N_UV)

def af2d_separable(u_grid, v_grid, Nx, Ny, dx, dy, u0, v0):
    def af1d(u, N, d, u0):
        psi = 2 * np.pi * d * (u - u0)
        with np.errstate(divide='ignore', invalid='ignore'):
            return np.where(np.abs(np.sin(psi / 2)) < 1e-12, 1.0,
                            np.abs(np.sin(N * psi / 2) / (N * np.sin(psi / 2))))
    return af1d(u_grid, Nx, dx, u0) * af1d(v_grid, Ny, dy, v0)

def draw_planar(change=None):
    uu, vv = np.meshgrid(u1d, u1d)
    mask = uu**2 + vv**2 <= 1.0
    pat = np.where(mask,
                   af2d_separable(uu, vv, w_2Nx.value, w_2Ny.value,
                                  w_2dx.value, w_2dy.value,
                                  w_2u0.value, w_2v0.value), np.nan)
    pat_dB = np.where(mask, 20 * np.log10(pat + 1e-10), np.nan)

    # E-plane cut (v = v0) and H-plane cut (u = u0)
    e_cut = af2d_separable(u1d, w_2v0.value * np.ones_like(u1d),
                           w_2Nx.value, w_2Ny.value,
                           w_2dx.value, w_2dy.value,
                           w_2u0.value, w_2v0.value)
    h_cut = af2d_separable(w_2u0.value * np.ones_like(u1d), u1d,
                           w_2Nx.value, w_2Ny.value,
                           w_2dx.value, w_2dy.value,
                           w_2u0.value, w_2v0.value)

    with out_2d:
        clear_output(wait=True)
        fig = plt.figure(figsize=(14, 5.5))
        ax1 = fig.add_subplot(121)
        ax2 = fig.add_subplot(122)

        im = ax1.imshow(pat_dB, origin='lower', extent=[-1, 1, -1, 1],
                        cmap='inferno', vmin=-30, vmax=0, aspect='equal')
        ax1.scatter([w_2u0.value], [w_2v0.value], marker='+', s=250,
                    color='cyan', zorder=5)
        ax1.add_patch(plt.Circle((0, 0), 1, color='white', fill=False, lw=1))
        # cut lines
        ax1.axhline(w_2v0.value, color='tomato', ls='--', lw=0.8, alpha=0.7)
        ax1.axvline(w_2u0.value, color='steelblue', ls='--', lw=0.8, alpha=0.7)
        ax1.set_xlabel('u = sinθ cosφ');  ax1.set_ylabel('v = sinθ sinφ')
        ax1.set_title(f'{w_2Nx.value}×{w_2Ny.value} array   '
                      f'steer ({w_2u0.value:.2f}, {w_2v0.value:.2f})')
        plt.colorbar(im, ax=ax1, fraction=0.046, label='AF (dB)')

        e_dB = 20 * np.log10(e_cut + 1e-10)
        h_dB = 20 * np.log10(h_cut + 1e-10)
        ax2.plot(u1d, e_dB, lw=2, color='tomato', label='E-plane (v=v₀)')
        ax2.plot(u1d, h_dB, lw=2, color='steelblue', label='H-plane (u=u₀)')
        ax2.set_xlim(-1, 1);  ax2.set_ylim(-40, 3)
        ax2.set_xlabel('direction cosine');  ax2.set_ylabel('AF (dB)')
        ax2.set_title('Principal plane cuts')
        ax2.legend(fontsize=9);  ax2.grid(alpha=0.3)

        plt.tight_layout();  plt.show()

for w in [w_2Nx, w_2Ny, w_2dx, w_2dy, w_2u0, w_2v0]:
    w.observe(draw_planar, names='value')
draw_planar()
display(VBox([HBox([w_2Nx, w_2Ny, w_2dx, w_2dy]),
              HBox([w_2u0, w_2v0]), out_2d]))

## 9 — Amplitude Tapering: Side-Lobe Control

Uniform weights give the narrowest main lobe but highest side lobes (−13.3 dB).  
Tapering the amplitude distribution trades beamwidth for lower side lobes:

| Window | First SLL | HPBW factor |
|--------|-----------|-------------|
| Uniform | −13.3 dB | 1.0× |
| Hamming | −41 dB | 1.5× |
| Taylor (−30 dB) | −30 dB | ~1.3× |
| Chebyshev (−40 dB) | −40 dB | ~1.4× |

Below: compare the array response and element weights for different windows,  
plus E/H field maps showing how tapering cleans up the wavefront.

In [ ]:
w_tN   = IntSlider(value=16, min=4, max=48, step=1,
                   description='Elements N', continuous_update=False,
                   style={'description_width': '90px'})
w_twin = Dropdown(options=['Uniform', 'Hamming', 'Hann', 'Blackman'],
                  value='Uniform', description='Window',
                  style={'description_width': '90px'})
w_td   = FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05,
                     description='d/λ', continuous_update=False,
                     style={'description_width': '90px'})
w_tth  = FloatSlider(value=0.0, min=-80, max=80, step=1,
                     description='Steer θ₀ (°)', continuous_update=False,
                     style={'description_width': '90px'})
out_tap = Output()

win_map = {'Uniform': np.ones, 'Hamming': np.hamming,
           'Hann': np.hanning, 'Blackman': np.blackman}

def draw_taper(change=None):
    N = w_tN.value;  d = w_td.value
    th0 = np.radians(w_tth.value)
    win = win_map[w_twin.value](N);  win /= win.max()
    ns = np.arange(N)
    k = 2 * np.pi
    delta = -k * d * np.sin(th0)

    th_plot = np.linspace(-90, 90, 3600)
    th_plot_rad = np.radians(th_plot)

    phase = ns[:, None] * (k * d * np.sin(th_plot_rad)[None, :] + delta)
    AF = np.abs(np.sum(win[:, None] * np.exp(1j * phase), axis=0))
    AF /= AF.max() + 1e-30
    AF_dB = 20 * np.log10(AF + 1e-10)

    # also uniform for comparison
    AF_uni = np.abs(np.sum(np.exp(1j * phase), axis=0))
    AF_uni /= AF_uni.max() + 1e-30
    AF_uni_dB = 20 * np.log10(AF_uni + 1e-10)

    # field map with taper
    lam = 1.0
    ys = (np.arange(N) - (N - 1) / 2) * d
    delays = -np.arange(N) * d * np.sin(th0) / lam
    xg = np.linspace(0.2, 8, 250)
    yg = np.linspace(ys.min() - 3, ys.max() + 3, 250)
    X, Y = np.meshgrid(xg, yg)
    field = np.zeros_like(X)
    for i, (yi, tau) in enumerate(zip(ys, delays)):
        R = np.sqrt(X**2 + (Y - yi)**2) + 1e-12
        field += win[i] * np.cos(2 * np.pi * (-tau) - k * R) / np.sqrt(R)

    # HPBW
    hp = np.where(AF_dB >= -3)[0]
    hpbw = th_plot[hp[-1]] - th_plot[hp[0]] if len(hp) >= 2 else np.nan
    # first SLL
    from scipy.signal import argrelmax
    pks = argrelmax(AF, order=20)[0]
    if len(pks) >= 2:
        main_pk = pks[np.argmax(AF[pks])]
        sides = [p for p in pks if p != main_pk]
        sll = 20 * np.log10(max(AF[sides])) if sides else -60
    else:
        sll = -60

    with out_tap:
        clear_output(wait=True)
        fig = plt.figure(figsize=(15, 9))
        ax1 = fig.add_subplot(221)
        ax2 = fig.add_subplot(222)
        ax3 = fig.add_subplot(223)
        ax4 = fig.add_subplot(224)

        # pattern comparison
        ax1.plot(th_plot, AF_dB, lw=2, color='gold', label=w_twin.value)
        ax1.plot(th_plot, AF_uni_dB, lw=1, color='gray', alpha=0.5, label='Uniform')
        ax1.axhline(-3, color='steelblue', ls=':', lw=0.8)
        ax1.set_xlim(-90, 90);  ax1.set_ylim(-60, 3)
        ax1.set_xlabel('θ (°)');  ax1.set_ylabel('AF (dB)')
        ax1.set_title(f'{w_twin.value}   HPBW={hpbw:.1f}°  SLL={sll:.1f} dB')
        ax1.legend(fontsize=8);  ax1.grid(alpha=0.3)

        # weights
        ax2.bar(ns, win, color='steelblue', alpha=0.8)
        ax2.set_xlabel('Element index');  ax2.set_ylabel('Weight')
        ax2.set_title(f'{w_twin.value} weights  N={N}')
        ax2.grid(alpha=0.3)

        # field map
        vmax = np.percentile(np.abs(field), 97)
        ax3.pcolormesh(X, Y, field, cmap='RdBu_r', shading='auto',
                       vmin=-vmax, vmax=vmax, rasterized=True)
        ax3.scatter(np.zeros(N), ys, marker='|', s=150, c='lime',
                    linewidths=2, zorder=5)
        ax3.set_xlabel('x / λ');  ax3.set_ylabel('y / λ')
        ax3.set_title(f'Wavefront with {w_twin.value} taper')
        ax3.set_aspect('equal')

        # intensity (dB)
        intensity = np.abs(field)**2
        intensity /= intensity.max() + 1e-30
        ax4.pcolormesh(X, Y, 10 * np.log10(intensity + 1e-10),
                       cmap='hot', shading='auto', vmin=-30, vmax=0,
                       rasterized=True)
        ax4.scatter(np.zeros(N), ys, marker='|', s=150, c='lime',
                    linewidths=2, zorder=5)
        ax4.set_xlabel('x / λ');  ax4.set_ylabel('y / λ')
        ax4.set_title('Intensity |E|² (dB)')
        ax4.set_aspect('equal')

        plt.tight_layout();  plt.show()

for w in [w_tN, w_twin, w_td, w_tth]:
    w.observe(draw_taper, names='value')
draw_taper()
display(VBox([HBox([w_tN, w_twin]), HBox([w_td, w_tth]), out_tap]))

## 10 — Null Steering & Adaptive Beamforming

Beyond steering the main beam, phased arrays can place **nulls** at specific angles to reject interference.  
Given a desired null at $\theta_j$, the weight vector is adjusted so that:
$$\mathbf{w}^H \mathbf{a}(\theta_j) = 0$$
where $\mathbf{a}(\theta) = [1,\, e^{jkd\sin\theta},\, \ldots,\, e^{j(N-1)kd\sin\theta}]^T$ is the steering vector.

Below: steer the main beam and place up to 2 nulls.  
Weights are computed via constrained optimisation (LCMV-style projection).

In [ ]:
w_aN  = IntSlider(value=12, min=4, max=24, step=1,
                  description='Elements N', continuous_update=False,
                  style={'description_width': '100px'})
w_ad  = FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05,
                    description='d/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_ath = FloatSlider(value=0.0, min=-80, max=80, step=1,
                    description='Beam θ₀ (°)', continuous_update=False,
                    style={'description_width': '100px'})
w_n1  = FloatSlider(value=-30.0, min=-80, max=80, step=1,
                    description='Null 1 (°)', continuous_update=False,
                    style={'description_width': '100px'})
w_n2  = FloatSlider(value=45.0, min=-80, max=80, step=1,
                    description='Null 2 (°)', continuous_update=False,
                    style={'description_width': '100px'})
w_null_on = Checkbox(value=True, description='Enable nulls',
                     style={'description_width': '100px'})
out_null = Output()

def steering_vector(theta_rad, N, d):
    ns = np.arange(N)
    return np.exp(1j * 2 * np.pi * d * ns * np.sin(theta_rad))

def draw_nullsteer(change=None):
    N = w_aN.value;  d = w_ad.value
    th0 = np.radians(w_ath.value)
    th_n1 = np.radians(w_n1.value)
    th_n2 = np.radians(w_n2.value)

    th_plot = np.linspace(-90, 90, 3600)
    th_plot_rad = np.radians(th_plot)

    # steering vectors
    a0 = steering_vector(th0, N, d)

    if w_null_on.value:
        a1 = steering_vector(th_n1, N, d)
        a2 = steering_vector(th_n2, N, d)
        # LCMV: C = [a0, a1, a2],  f = [1, 0, 0]
        C = np.column_stack([a0, a1, a2])
        f = np.array([1, 0, 0])
        # w = C (C^H C)^{-1} f  →  w^H a(θ_j) = f_j
        CTC_inv = np.linalg.inv(C.conj().T @ C)
        weights = C @ CTC_inv @ f
    else:
        weights = a0.conj() / N

    # compute pattern: response = w^H a(θ)
    pat = np.zeros(len(th_plot), dtype=complex)
    for i in range(N):
        pat += weights[i].conj() * np.exp(1j * 2 * np.pi * d * i * np.sin(th_plot_rad))
    pat_abs = np.abs(pat)
    pat_abs /= pat_abs.max() + 1e-30
    pat_dB = 20 * np.log10(pat_abs + 1e-10)

    with out_null:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(th_plot, pat_dB, lw=2, color='gold')
        ax1.axvline(w_ath.value, color='lime', ls=':', lw=1.5, label=f'Beam {w_ath.value:.0f}°')
        if w_null_on.value:
            ax1.axvline(w_n1.value, color='tomato', ls='--', lw=1.5, label=f'Null {w_n1.value:.0f}°')
            ax1.axvline(w_n2.value, color='cyan', ls='--', lw=1.5, label=f'Null {w_n2.value:.0f}°')
        ax1.set_xlim(-90, 90);  ax1.set_ylim(-60, 3)
        ax1.set_xlabel('θ (°)');  ax1.set_ylabel('AF (dB)')
        ax1.set_title('Adaptive pattern with nulls')
        ax1.legend(fontsize=8);  ax1.grid(alpha=0.3)

        # weight amplitudes and phases
        ax2_twin = ax2.twinx()
        ax2.bar(np.arange(N) - 0.15, np.abs(weights), width=0.3,
                color='steelblue', alpha=0.8, label='|w|')
        ax2_twin.bar(np.arange(N) + 0.15, np.degrees(np.angle(weights)),
                     width=0.3, color='tomato', alpha=0.6, label='∠w (°)')
        ax2.set_xlabel('Element');  ax2.set_ylabel('|w|', color='steelblue')
        ax2_twin.set_ylabel('∠w (°)', color='tomato')
        ax2.set_title('Element weights')
        lines1, labels1 = ax2.get_legend_handles_labels()
        lines2, labels2 = ax2_twin.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
        ax2.grid(alpha=0.3)

        plt.tight_layout();  plt.show()

for w in [w_aN, w_ad, w_ath, w_n1, w_n2, w_null_on]:
    w.observe(draw_nullsteer, names='value')
draw_nullsteer()
display(VBox([HBox([w_aN, w_ad, w_ath]),
              HBox([w_n1, w_n2, w_null_on]), out_null]))